In [17]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Load dataset
data_path = 'Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)

# Drop 'customerID' as it's not relevant
df.drop(columns=['customerID'], inplace=True)

# Convert target variable to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Convert 'TotalCharges' to numeric, handling errors as NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Select relevant features
selected_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'Contract', 'PaymentMethod',
                     'PaperlessBilling', 'PhoneService', 'InternetService', 'OnlineSecurity', 'TechSupport']

# Prepare feature matrix and target variable
X = df[selected_features]
y = df['Churn']

# Identify numerical and categorical features
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = ['gender', 'Contract', 'PaymentMethod', 'PaperlessBilling', 'PhoneService', 'InternetService', 'OnlineSecurity', 'TechSupport']

# Define preprocessing pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply transformations
X_train = full_pipeline.fit_transform(X_train)
X_test = full_pipeline.transform(X_test)

# Define classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB()
}

# Train, evaluate, and store results
results = {}

for name, model in classifiers.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)

    results[name] = {"accuracy": accuracy, "roc_auc": roc_auc, "model": model}
    print(f"{name} → Accuracy: {accuracy:.4f}, ROC AUC: {roc_auc:.4f}")

# Select the best model based on ROC AUC (or accuracy if tied)
best_model_name = max(results, key=lambda x: results[x]["roc_auc"])
best_model = results[best_model_name]["model"]

print(f"\n Best Model: {best_model_name} → ROC AUC: {results[best_model_name]['roc_auc']:.4f}")

# Save the best model and preprocessor
with open("model.pkl", "wb") as file:
    pickle.dump(best_model, file)

with open("preprocessor.pkl", "wb") as file:
    pickle.dump(full_pipeline, file)


Logistic Regression → Accuracy: 0.7963, ROC AUC: 0.7179
K-Nearest Neighbors → Accuracy: 0.7835, ROC AUC: 0.6964
Random Forest → Accuracy: 0.7828, ROC AUC: 0.6968
Decision Tree → Accuracy: 0.7480, ROC AUC: 0.6705
Naive Bayes → Accuracy: 0.7119, ROC AUC: 0.7356

 Best Model: Naive Bayes → ROC AUC: 0.7356
